In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import tensorflow as tf
from tensorflow.keras import layers,Model

class EncoderLayer(tf.keras.layers.Layer):

    def __init__(self,d_model,num_heads,dff,dropout_rate=0.1,**kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.dff = dff
        self.dropout_rate = dropout_rate

        # MultiheadAttention
        self.mha = layers.MultiHeadAttention(num_heads=num_heads,
                                            key_dim = d_model//num_heads,
                                            name='self_attention')
        self.ffn_dense1 = layers.Dense(dff,activation='relu',name='ffn_dense1')
        self.ffn_dense2 = layers.Dense(d_model, name = 'ffn_dense2')
        self.norm1 = layers.LayerNormalization(epsilon=1e-6,name='norm1')
        self.norm2 = layers.LayerNormalization(epsilon=1e-6,name='norm2')
        #dropout before residual
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self,x,training=False,mask=None):
        attn_output = self.mha(x,x,x,attention_mask=mask)
        attn_output = self.dropout1(attn_output,training=training)
        out1 = self.norm1(x+attn_output)
        ffn_output = self.ffn_dense2(self.ffn_dense1(out1))
        ffn_output = self.dropout2(ffn_output,training=training)
        out2 = self.norm2(out1+ffn_output)
        return out2

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model':self.d_model,'num_heads':self.num_heads,
            'dff':self.dff,'droput_rate': self.dropout_rate
        })
        return config

D_MODEL =32
NUM_HEADS=2
DFF=64
SEQ_LEN=60

enc_input = layers.Input(shape=(SEQ_LEN,D_MODEL),name='encoder_input')
enc_output = EncoderLayer(D_MODEL,NUM_HEADS,DFF,name='encoder_layer')(enc_input)
encoder_layer_model = Model(inputs=enc_input,outputs=enc_output,name='EncoderLayerModel')
print("=====Encoder Model Summary=====")
encoder_layer_model.summary()

=====Encoder Model Summary=====


2026-09-14 17:40:51.311364: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "EncoderLayerModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 60, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer (EncoderLayer)    │ (None, 60, 32)         │         8,544 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,544 (33.38 KB)

 Trainable params: 8,544 (33.38 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self,d_model,num_heads,dff,dropout_rate=0.1,**kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.droput_rate = dropout_rate
        self.dff =dff
        self.mha1 = layers.MultiHeadAttention(num_heads = num_heads,key_dim=d_model//num_heads,
                                             name='masked_self_attention')
        self.mha2 = layers.MultiHeadAttention(num_heads=num_heads,key_dim=d_model//num_heads,
                                             name='cross_attention')
        self.ffn_dense1 = layers.Dense(dff,activation='relu',name='ffn_dense1')
        self.ffn_dense2 = layers.Dense(d_model,name='ffn_dense2')
        self.norm1 = layers.LayerNormalization(epsilon=1e-6,name='norm1')
        self.norm2 = layers.LayerNormalization(epsilon=1e-6,name='norm2')
        self.norm3 = layers.LayerNormalization(epsilon=1e-6,name='norm3')
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.dropout3 = layers.Dropout(dropout_rate)

    def call(self,x,enc_out,training=False,look_ahead_mask=None,padding_mask=None):

        attn1 = self.mha1(x,x,x,attention_mask = look_ahead_mask)#self attention
        attn1 = self.dropout1(attn1,training=training)
        out1 = self.norm1(x+attn1)

        attn2 = self.mha2(out1,enc_out,enc_out,attention_mask=padding_mask)#encoder-decoder attention/cross
        attn2 = self.dropout2(attn2,training=training)
        out2 = self.norm2(out1+attn2)

        ffn_out = self.ffn_dense2(self.ffn_dense1(out2))
        ffn_out = self.dropout3(ffn_out,training=training)
        out3 = self.norm3(out2+ffn_out)
        return out3

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model, "num_heads": self.num_heads,
            "dff": self.dff, "dropout_rate": self.dropout_rate
        })
        return config

tgt_input = layers.Input(shape=(SEQ_LEN,D_MODEL),name='target_input')
src_input = layers.Input(shape=(SEQ_LEN,D_MODEL),name = 'encoder_output')
dec_layer = DecoderLayer(D_MODEL,NUM_HEADS,DFF,name='decoder_layer')
dec_out = dec_layer(tgt_input,src_input)
decoder_layer_model = Model(inputs=[tgt_input,src_input],outputs=dec_out,
                           name='DecoderLayer_Model')
print("======DecoderLayer Model Summary=======")
decoder_layer_model.summary()

======DecoderLayer Model Summary=======


Model: "DecoderLayer_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ target_input        │ (None, 60, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_output      │ (None, 60, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer       │ (None, 60, 32)    │     12,832 │ target_input[0][… │
│ (DecoderLayer)      │                   │            │ encoder_output[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,832 (50.12 KB)

 Trainable params: 12,832 (50.12 KB)

 Non-trainable params: 0 (0.00 B)